# 09 Wetterverteilung analysieren

Dieses Notebook implementiert BD-19. Es prüft auf Match-Ebene, ob Temperatur, gefühlte Temperatur und Regen genug Streuung für die folgenden Analysefragen haben.

Input:
- `data/gold/team_match_analysis_dataset.csv`

Outputs:
- `outputs/figures/weather_temperature_histogram.png`
- `outputs/figures/weather_feels_like_histogram.png`
- `outputs/figures/weather_temperature_by_tournament.png`
- `outputs/figures/rain_share.png`
- `outputs/tables/bd19_weather_distribution_summary.csv`

## Methodischer Ansatz

Das Gold-Dataset enthält pro Spiel zwei Team-Perspektiven. Für Wetterverteilungen wird deshalb zuerst auf eindeutige `match_id` reduziert, damit jedes Wetterereignis nur einmal zählt.

In [ ]:
from pathlib import Path
import math

import pandas as pd
from PIL import Image, ImageDraw, ImageFont

from project_paths import project_root

ROOT = project_root()
DATA_PATH = ROOT / 'data' / 'gold' / 'team_match_analysis_dataset.csv'
MODEL_COMPARISON_PATH = ROOT / 'outputs' / 'tables' / 'model_comparison.csv'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    'ink': '#222222',
    'muted': '#555f73',
    'grid': '#d6d6d6',
    'axis': '#c9c9c9',
    'bg': '#ffffff',
    'panel': '#f5f5f5',
    'blue': '#6679a8',
    'orange': '#c98d66',
    'green': '#6ca176',
    'teal': '#5c9a93',
    'red': '#b25f64',
    'purple': '#8b7caf',
    'pink': '#c791bb',
    'gray': '#8f8f8f',
    'olive': '#c0b783',
    'cyan': '#7fb0c0',
    'gold': '#d9a332',
}
COLOR_CYCLE = ['blue', 'orange', 'green', 'red', 'purple', 'pink', 'gray', 'olive', 'cyan']


def _font(size=28, bold=False):
    candidates = [
        'C:/Windows/Fonts/arialbd.ttf' if bold else 'C:/Windows/Fonts/arial.ttf',
        'C:/Windows/Fonts/segoeuib.ttf' if bold else 'C:/Windows/Fonts/segoeui.ttf',
    ]
    for candidate in candidates:
        if Path(candidate).exists():
            return ImageFont.truetype(candidate, size)
    return ImageFont.load_default()

FONT_TITLE = _font(34, True)
FONT_SUBTITLE = _font(20)
FONT_LABEL = _font(19)
FONT_SMALL = _font(16)
FONT_TINY = _font(14)
FONT_NUMBER = _font(36, True)


def _hex(color):
    color = color.lstrip('#')
    return tuple(int(color[i:i+2], 16) for i in (0, 2, 4))


def _text(draw, xy, text, fill='ink', font=None, anchor=None):
    draw.text(xy, str(text), fill=_hex(PALETTE.get(fill, fill)), font=font or FONT_LABEL, anchor=anchor)


def _canvas(title, subtitle=None, size=(1600, 900)):
    image = Image.new('RGB', size, _hex(PALETTE['bg']))
    draw = ImageDraw.Draw(image)
    _text(draw, (70, 46), title, font=FONT_TITLE)
    if subtitle:
        _text(draw, (70, 92), subtitle, fill='muted', font=FONT_SUBTITLE)
    return image, draw


def _save(image, path):
    image.save(path, quality=95)
    return path


def _nice_max(value):
    if value <= 0:
        return 1
    exp = math.floor(math.log10(value))
    base = 10 ** exp
    for mult in [1, 2, 5, 10]:
        if value <= mult * base:
            return mult * base
    return 10 * base


def _draw_plot_frame(draw, left, top, right, bottom):
    draw.rectangle([left, top, right, bottom], outline=_hex(PALETTE['axis']), width=1)


def draw_histogram(series, path, title, subtitle, xlabel, bins=14, color='blue'):
    values = pd.Series(series).dropna().astype(float)
    image, draw = _canvas(title, subtitle)
    left, top, right, bottom = 170, 185, 1510, 730
    min_v, max_v = float(values.min()), float(values.max())
    width = (max_v - min_v) / bins
    counts = []
    for i in range(bins):
        lo = min_v + i * width
        hi = min_v + (i + 1) * width
        if i == bins - 1:
            count = int(((values >= lo) & (values <= hi)).sum())
        else:
            count = int(((values >= lo) & (values < hi)).sum())
        counts.append(count)
    y_max = _nice_max(max(counts))
    _draw_plot_frame(draw, left, top, right, bottom)
    _text(draw, (left, top - 28), 'Anzahl Spiele', fill='muted', font=FONT_TINY)
    for tick in range(0, 6):
        y = bottom - (bottom - top) * tick / 5
        draw.line([(left, y), (right, y)], fill=_hex(PALETTE['grid']), width=1)
        _text(draw, (left - 16, y), f'{int(y_max * tick / 5)}', fill='muted', font=FONT_TINY, anchor='rm')
    bar_gap = 4
    bar_w = (right - left) / bins
    for i, count in enumerate(counts):
        x0 = left + i * bar_w + bar_gap
        x1 = left + (i + 1) * bar_w - bar_gap
        y0 = bottom - (bottom - top) * count / y_max
        draw.rectangle([x0, y0, x1, bottom], fill=_hex(PALETTE[color]))
    median = float(values.median())
    median_x = left + (median - min_v) / (max_v - min_v) * (right - left)
    draw.line([(median_x, top), (median_x, bottom)], fill=_hex(PALETTE['red']), width=3)
    _text(draw, (median_x + 8, top + 9), f'Median {median:.1f}', fill='red', font=FONT_TINY)
    for i in range(0, 6):
        x = left + (right - left) * i / 5
        value = min_v + (max_v - min_v) * i / 5
        _text(draw, (x, bottom + 22), f'{value:.0f}', fill='muted', font=FONT_TINY, anchor='mt')
    _text(draw, ((left + right) / 2, 810), xlabel, fill='ink', font=FONT_SMALL, anchor='mm')
    _text(draw, (left, 140), f'n={len(values)} | min={min_v:.1f} | max={max_v:.1f}', fill='muted', font=FONT_TINY)
    return _save(image, path)


def draw_boxplot(groups, path, title, subtitle, ylabel, color='blue', value_fmt='{:.1f}', y_min=None, y_max=None):
    clean = [(label, pd.Series(vals).dropna().astype(float)) for label, vals in groups if len(pd.Series(vals).dropna())]
    all_values = pd.concat([vals for _, vals in clean])
    y_min = float(all_values.min()) if y_min is None else y_min
    y_max = float(all_values.max()) if y_max is None else y_max
    pad = (y_max - y_min) * 0.08 or 1
    y_min, y_max = y_min - pad, y_max + pad
    image, draw = _canvas(title, subtitle)
    left, top, right, bottom = 170, 185, 1510, 730
    _draw_plot_frame(draw, left, top, right, bottom)
    _text(draw, (left, top - 28), ylabel, fill='muted', font=FONT_TINY)
    for tick in range(0, 6):
        value = y_min + (y_max - y_min) * tick / 5
        y = bottom - (value - y_min) / (y_max - y_min) * (bottom - top)
        draw.line([(left, y), (right, y)], fill=_hex(PALETTE['grid']), width=1)
        _text(draw, (left - 16, y), value_fmt.format(value), fill='muted', font=FONT_TINY, anchor='rm')
    n = len(clean)
    step = (right - left) / n
    for idx, (label, vals) in enumerate(clean):
        q1, med, q3 = vals.quantile([0.25, 0.5, 0.75])
        low, high = vals.min(), vals.max()
        x = left + step * (idx + 0.5)
        box_w = min(120, step * 0.42)
        cap_w = box_w * 0.50
        fill_color = COLOR_CYCLE[idx % len(COLOR_CYCLE)] if color == 'cycle' else color
        def y(value):
            return bottom - (float(value) - y_min) / (y_max - y_min) * (bottom - top)
        draw.line([(x, y(low)), (x, y(high))], fill=_hex(PALETTE['ink']), width=2)
        draw.line([(x - cap_w / 2, y(low)), (x + cap_w / 2, y(low))], fill=_hex(PALETTE['ink']), width=2)
        draw.line([(x - cap_w / 2, y(high)), (x + cap_w / 2, y(high))], fill=_hex(PALETTE['ink']), width=2)
        draw.rectangle([x - box_w / 2, y(q3), x + box_w / 2, y(q1)], fill=_hex(PALETTE[fill_color]), outline=_hex(PALETTE['ink']), width=1)
        draw.line([(x - box_w / 2, y(med)), (x + box_w / 2, y(med))], fill=_hex(PALETTE['gold']), width=4)
        _text(draw, (x, y(med) - 10), value_fmt.format(float(med)), fill='ink', font=FONT_TINY, anchor='mb')
        _text(draw, (x, bottom + 24), label, fill='ink', font=FONT_SMALL, anchor='mt')
        _text(draw, (x, bottom + 48), f'n={len(vals)}', fill='muted', font=FONT_TINY, anchor='mt')
    return _save(image, path)


def draw_bar_chart(items, path, title, subtitle, ylabel='', color='blue', percent=False):
    image, draw = _canvas(title, subtitle)
    left, top, right, bottom = 170, 185, 1510, 730
    values = [float(item[1]) for item in items]
    y_max = _nice_max(max(values) * 1.2)
    _draw_plot_frame(draw, left, top, right, bottom)
    if ylabel:
        _text(draw, (left, top - 28), ylabel, fill='muted', font=FONT_TINY)
    for tick in range(0, 6):
        value = y_max * tick / 5
        y = bottom - (bottom - top) * tick / 5
        draw.line([(left, y), (right, y)], fill=_hex(PALETTE['grid']), width=1)
        if percent:
            tick_label = f'{value:.0f}%'
        elif y_max <= 1:
            tick_label = f'{value:.2f}'
        else:
            tick_label = f'{value:.0f}'
        _text(draw, (left - 16, y), tick_label, fill='muted', font=FONT_TINY, anchor='rm')
    step = (right - left) / len(items)
    bar_w = min(125, step * 0.5)
    for idx, (label, value) in enumerate(items):
        x = left + step * (idx + 0.5)
        y = bottom - (bottom - top) * float(value) / y_max
        fill_color = COLOR_CYCLE[idx % len(COLOR_CYCLE)] if color == 'cycle' else color
        draw.rectangle([x - bar_w / 2, y, x + bar_w / 2, bottom], fill=_hex(PALETTE[fill_color]))
        text_value = f'{value:.1f}%' if percent else f'{value:.3f}' if value < 1 else f'{value:.1f}'
        _text(draw, (x, y - 12), text_value, fill='ink', font=FONT_SMALL, anchor='mb')
        _text(draw, (x, bottom + 25), label, fill='ink', font=FONT_SMALL, anchor='mt')
    return _save(image, path)


def draw_two_panel_boxplot(left_groups, right_groups, path, title, subtitle):
    image, draw = _canvas(title, subtitle)
    panels = [
        (left_groups, (135, 215, 760, 710), 'Passquote nach Temperaturgruppe', 'Passquote', '{:.0f}%'),
        (right_groups, (900, 215, 1515, 710), 'Long-Ball-Share nach Regen', 'Long-Ball-Share', '{:.0f}%'),
    ]
    for panel_idx, (groups, area, panel_title, y_label, fmt) in enumerate(panels):
        left, top, right, bottom = area
        clean = [(label, pd.Series(vals).dropna().astype(float) * 100) for label, vals in groups if len(pd.Series(vals).dropna())]
        all_values = pd.concat([vals for _, vals in clean])
        y_min, y_max = float(all_values.min()), float(all_values.max())
        pad = (y_max - y_min) * 0.12 or 1
        y_min, y_max = y_min - pad, y_max + pad
        _text(draw, (left, top - 42), panel_title, font=FONT_LABEL)
        _text(draw, (left, top - 18), y_label, fill='muted', font=FONT_TINY)
        _draw_plot_frame(draw, left, top, right, bottom)
        for tick in range(0, 5):
            value = y_min + (y_max - y_min) * tick / 4
            y = bottom - (value - y_min) / (y_max - y_min) * (bottom - top)
            draw.line([(left, y), (right, y)], fill=_hex(PALETTE['grid']), width=1)
            _text(draw, (left - 12, y), fmt.format(value), fill='muted', font=FONT_TINY, anchor='rm')
        n = len(clean)
        step = (right - left) / n
        for idx, (label, vals) in enumerate(clean):
            q1, med, q3 = vals.quantile([0.25, 0.5, 0.75])
            low, high = vals.min(), vals.max()
            x = left + step * (idx + 0.5)
            box_w = min(105, step * 0.42)
            cap_w = box_w * 0.50
            fill_color = COLOR_CYCLE[(panel_idx * 3 + idx) % len(COLOR_CYCLE)]
            def y(value):
                return bottom - (float(value) - y_min) / (y_max - y_min) * (bottom - top)
            draw.line([(x, y(low)), (x, y(high))], fill=_hex(PALETTE['ink']), width=2)
            draw.line([(x - cap_w / 2, y(low)), (x + cap_w / 2, y(low))], fill=_hex(PALETTE['ink']), width=2)
            draw.line([(x - cap_w / 2, y(high)), (x + cap_w / 2, y(high))], fill=_hex(PALETTE['ink']), width=2)
            draw.rectangle([x - box_w / 2, y(q3), x + box_w / 2, y(q1)], fill=_hex(PALETTE[fill_color]), outline=_hex(PALETTE['ink']), width=1)
            draw.line([(x - box_w / 2, y(med)), (x + box_w / 2, y(med))], fill=_hex(PALETTE['gold']), width=4)
            _text(draw, (x, y(med) - 10), fmt.format(float(med)), fill='ink', font=FONT_TINY, anchor='mb')
            _text(draw, (x, bottom + 24), label, fill='ink', font=FONT_SMALL, anchor='mt')
            _text(draw, (x, bottom + 48), f'n={len(vals)}', fill='muted', font=FONT_TINY, anchor='mt')
    return _save(image, path)

TEMPERATURE_HIST_PATH = FIGURE_DIR / 'weather_temperature_histogram.png'
FEELS_LIKE_HIST_PATH = FIGURE_DIR / 'weather_feels_like_histogram.png'
TEMPERATURE_BY_TOURNAMENT_PATH = FIGURE_DIR / 'weather_temperature_by_tournament.png'
RAIN_SHARE_PATH = FIGURE_DIR / 'rain_share.png'
SUMMARY_PATH = TABLE_DIR / 'bd19_weather_distribution_summary.csv'

required_columns = [
    'match_id', 'short_name', 'temperature_c', 'feels_like_c',
    'rain_mm', 'rain_flag', 'weather_missing_flag',
]

## Daten laden und Match-Ebene bilden

Die Wetterdaten werden aus dem finalen Analyse-Dataset gelesen. Durch `drop_duplicates('match_id')` bleibt pro Spiel genau eine Wetterbeobachtung erhalten.

In [ ]:
assert DATA_PATH.exists(), f'Missing required input file: {DATA_PATH}'

team_match_df = pd.read_csv(DATA_PATH)
missing_columns = [column for column in required_columns if column not in team_match_df.columns]
assert not missing_columns, f'Missing required columns: {missing_columns}'

weather_df = (
    team_match_df[required_columns]
    .drop_duplicates(subset=['match_id'])
    .sort_values(['short_name', 'match_id'])
    .reset_index(drop=True)
)

assert len(weather_df) > 0, 'Weather dataset is empty after reducing to match level.'
assert weather_df['match_id'].is_unique, 'Expected one row per match after deduplication.'
assert not weather_df['weather_missing_flag'].any(), 'Weather data still contains missing weather rows.'

weather_df.head()

## Kennzahlen zur Wetterstreuung

Die Tabelle fasst zentrale Streuungsmaße zusammen. Sie hilft bei der Entscheidung, ob Wetter als Kontextfaktor sinnvoll weiter analysiert werden kann.

In [ ]:
summary_rows = []
for column in ['temperature_c', 'feels_like_c', 'rain_mm']:
    series = weather_df[column].dropna()
    summary_rows.append({
        'metric': column,
        'n_matches': int(series.count()),
        'min': round(float(series.min()), 2),
        'q25': round(float(series.quantile(0.25)), 2),
        'median': round(float(series.median()), 2),
        'mean': round(float(series.mean()), 2),
        'q75': round(float(series.quantile(0.75)), 2),
        'max': round(float(series.max()), 2),
        'std': round(float(series.std()), 2),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_PATH, index=False)
summary

## Wetterplots

Die Plots zeigen Temperatur, gefühlte Temperatur, Turnierunterschiede und Regenanteile auf Match-Ebene.

In [ ]:
draw_histogram(
    weather_df['temperature_c'],
    TEMPERATURE_HIST_PATH,
    'Verteilung der Temperatur bei Anpfiff',
    'Die Spiele decken milde, warme und heiße Bedingungen ab.',
    'Temperatur in °C',
    bins=14,
    color='blue',
)

draw_histogram(
    weather_df['feels_like_c'],
    FEELS_LIKE_HIST_PATH,
    'Verteilung der gefühlten Temperatur bei Anpfiff',
    'Gefühlte Temperatur ergänzt die gemessene Temperatur als Kontextsignal.',
    'Gefühlte Temperatur in °C',
    bins=14,
    color='cyan',
)

tournament_order = list(weather_df['short_name'].drop_duplicates())
draw_boxplot(
    [(tournament, weather_df.loc[weather_df['short_name'] == tournament, 'temperature_c']) for tournament in tournament_order],
    TEMPERATURE_BY_TOURNAMENT_PATH,
    'Temperatur nach Turnier',
    'Turniere unterscheiden sich sichtbar im Wetterkontext.',
    'Temperatur in °C',
    color='cycle',
)

rain_by_tournament = (
    weather_df.groupby('short_name', sort=False)['rain_flag']
    .mean()
    .mul(100)
    .reset_index(name='rain_share_pct')
)
draw_bar_chart(
    list(zip(rain_by_tournament['short_name'], rain_by_tournament['rain_share_pct'])),
    RAIN_SHARE_PATH,
    'Anteil Spiele mit Regen je Turnier',
    f'Regen ist vorhanden, aber selten: insgesamt {weather_df["rain_flag"].mean() * 100:.1f}% der Matches.',
    ylabel='Spiele mit Regen',
    color='cycle',
    percent=True,
)

## Output-Checks

Die Checks stellen sicher, dass alle geforderten BD-19-Artefakte geschrieben wurden.

In [ ]:
figure_paths = [
    TEMPERATURE_HIST_PATH,
    FEELS_LIKE_HIST_PATH,
    TEMPERATURE_BY_TOURNAMENT_PATH,
    RAIN_SHARE_PATH,
]

for path in figure_paths:
    assert path.exists(), f'Missing expected figure: {path}'
    assert path.stat().st_size > 1_000, f'Figure looks too small or empty: {path}'

pd.DataFrame({
    'figure': [str(path.relative_to(ROOT)) for path in figure_paths],
    'bytes': [path.stat().st_size for path in figure_paths],
})

## Interpretation für BD-19

Die Temperaturwerte reichen von milden bis heißen Bedingungen und unterscheiden sich sichtbar zwischen Turnieren. Regen kommt seltener vor als trockene Spiele, ist aber in mehreren Turnieren vorhanden. Damit ist Wetter als Kontextvariable sinnvoll, Regenvergleiche sollten wegen der kleineren Gruppe aber vorsichtig interpretiert werden.

In [ ]:
interpretation = {
    'n_matches': int(len(weather_df)),
    'temperature_min_c': round(float(weather_df['temperature_c'].min()), 1),
    'temperature_median_c': round(float(weather_df['temperature_c'].median()), 1),
    'temperature_max_c': round(float(weather_df['temperature_c'].max()), 1),
    'feels_like_min_c': round(float(weather_df['feels_like_c'].min()), 1),
    'feels_like_median_c': round(float(weather_df['feels_like_c'].median()), 1),
    'feels_like_max_c': round(float(weather_df['feels_like_c'].max()), 1),
    'rain_match_share_pct': round(float(weather_df['rain_flag'].mean() * 100), 1),
    'rain_matches': int(weather_df['rain_flag'].sum()),
}
interpretation